In [32]:
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [29]:
# We use the Bi-Encoder to encode all passages, so that we can use it with semantic search

bi_encoder = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
bi_encoder.max_seq_length = 256  # Truncate long passages to 256 tokens
top_k = 32  # Number of passages we want to retrieve with the bi-encoder

# The bi-encoder will retrieve top_k documents. We use a cross-encoder to re-rank the results
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

# Load PDF and extract passages
file_path = "/home/senacgoon.local/202473567/Documents/senac_ia/uc15/senac_ia_uc15_nlp_project/data/pablo/sindilojas_2025_2026.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    add_start_index=True,
)

# split_documents returns Document objects; the encoders expect plain strings
chunk_docs = text_splitter.split_documents(docs)
passages = [doc.page_content.strip() for doc in chunk_docs if doc.page_content and doc.page_content.strip()]

print("Pages:", len(docs))
print("Passages:", len(passages))

# We encode all passages into our vector space
corpus_embeddings = bi_encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11538.20it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14793.04it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pages: 38
Passages: 225


Batches: 100%|██████████| 8/8 [00:00<00:00, 70.68it/s]


In [30]:
def search(query, top_n=3):
    print("Input question:", query)

    ##### Semantic Search #####
    # Encode the query using the bi-encoder and find potentially relevant passages
    question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
    question_embedding = question_embedding.to(corpus_embeddings.device)
    hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
    hits = hits[0]  # Get the hits for the first query

    ##### Re-Ranking #####
    # Score all retrieved passages with the cross_encoder
    cross_inp = [[query, passages[hit["corpus_id"]]] for hit in hits]
    cross_scores = cross_encoder.predict(cross_inp)

    # Attach cross-encoder score to each hit
    for idx, score in enumerate(cross_scores):
        hits[idx]["cross-score"] = float(score)

    # Output of top hits from bi-encoder
    print("\n-------------------------\n")
    print(f"Top-{top_n} Bi-Encoder Retrieval hits")
    hits_by_bi = sorted(hits, key=lambda x: x["score"], reverse=True)
    for hit in hits_by_bi[:top_n]:
        passage_text = passages[hit["corpus_id"]].replace("\n", " ")
        print("\t{:.3f}\t{}".format(hit["score"], passage_text))

    # Output of top hits from re-ranker
    print("\n-------------------------\n")
    print(f"Top-{top_n} Cross-Encoder Re-ranker hits")
    hits_by_cross = sorted(hits, key=lambda x: x["cross-score"], reverse=True)
    for hit in hits_by_cross[:top_n]:
        passage_text = passages[hit["corpus_id"]].replace("\n", " ")
        print("\t{:.3f}\t{}".format(hit["cross-score"], passage_text))

    return hits_by_cross[:top_n]

In [31]:
search(query="Quais as garantias ao retornar de férias?")

Input question: Quais as garantias ao retornar de férias?

-------------------------

Top-3 Bi-Encoder Retrieval hits
	0.559	infratora multa de R$ 712,00 (setecentos e doze reais) por empregado.    42 - GARANTIA DE EMPREGO APÓS RETORNO DO AUXÍLIO DOENÇA -  Ao comerciário  que retorna ao trabalho em razão de afastamento por doença, fica assegurada a manutenção  de seu contrato de trabalho pelo período de 30 (trinta) dias, a partir da alta previdenciária.     Parágrafo único - A garantia prevista nesta cláusula poderá ser substituída por indenização   correspondente aos dias ainda não implementados do período da garant ia, com acréscimo
	0.557	período de férias para efeito de cálculo do terço adicional, demais incidências e estabilidade.    g) independentemente da jornada, as empresas que têm cozinha e refeitórios  próprios, e  fornecem refeições, nos termos do PAT, fornecerão alimentação nesses dias ou, fora dessas  situações, fornecerão documento refeição ou indenização em dinheiro, co

[{'corpus_id': 127,
  'score': 0.5492233037948608,
  'cross-score': 1.0608878135681152},
 {'corpus_id': 110,
  'score': 0.4941350817680359,
  'cross-score': -2.1433887481689453},
 {'corpus_id': 126,
  'score': 0.559334397315979,
  'cross-score': -2.307190418243408}]